In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import time

from __future__ import annotations
from yandex_cloud_ml_sdk import YCloudML

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)

In [2]:
X_test = pd.read_csv('X_test_corr.csv',index_col=0)

In [3]:
X_test.head(10)

,text
0,назначение платежа: Перечисление средств по соглашению №2140 о предоставлении гранта победителю конкурса на предоставление грантов Раиса РТ на развитие гражданского общества от 26.11.2024г. НДС не облагается.\nназвание статьи: Грантик\nназвание проекта: Донорство в Татарстане\nкатегория донора: отсутствует
1,назначение платежа: субсидия за февраль 2025\nназвание статьи: Социальная дем.\nназвание проекта: Присмотр и уход\nкатегория донора: отсутствует
2,назначение платежа: (93810061540906600633246=2220593;02452000150)Субс-я на фин. обеспеч-е затрат соц. ориентир. некомер. организ-и в 2022 единовременно в соответ-и с граф-м перечисл. согл-е NГ-22 от 01.04.2022г. Без НДС\nназвание статьи: Гран\nназвание проекта: Станем сильнее вместе\nкатегория донора: Гран губернатора ЛО-2022-2023
3,назначение платежа: Перечисление средств для открытия депозита по депозитной сделке № UNV-0000001249512 от 18.03.2024 согласно заявлению клиента. НДС не предусмотрен.\nназвание статьи: гранти\nназвание проекта: игрец\nкатегория донора: средние
4,назначение платежа: Перечисление пожертвования по договору № Т-Гранты/НКО-25-000912 от 01.09.2025.НДС не облагается.\nназвание статьи: Гран\nназвание проекта: Административные расходы\nкатегория донора: Девелопмент Групп
5,назначение платежа: Перечисление средств по договору о предоставлении гранта Президента Российской Федерации на развитие гражданского общества от 30.01.2024г. № 24-1-004569. НДС не облагается.\nназвание статьи: Грантик\nназвание проекта: Решаем вместе\nкатегория донора: ФПГ_Решаем вместе
6,"назначение платежа: Оплата по договору о предоставлении гранта №Р36-24-1-000054 от 23.07.2024 в целях реализации проекта ""Раскрась жизнь""Сумма 886818-00Без налога (НДС)\nназвание статьи: Гран\nназвание проекта: Семейные выходные\nкатегория донора: ОБРАЗ БУДУЩЕГО"
7,назначение платежа: Перечисление средств (2) по договору о предоставлении гранта Президента Российской Федерации на развитие гражданского общества от 04.02.2025г. № 25-1-008456. НДС не облагается.\nназвание статьи: Грант или субсидия\nназвание проекта: отсутствует\nкатегория донора: Грантоотдающие организации
8,"назначение платежа: (148100204Д1000200631000л/с 02732592000.):1 541 506,46(24B КОСГУ) Оплата Дог. № 94 от 09.02.2024 акт б/н от 05.08.25 отчет за июль Б/О № 25574 возм.зат.за ок.адр.соц.пом.,НДС не облагается\nназвание статьи: Соц.услуги\nназвание проекта: Э_надо_разобрать\nкатегория донора: Соцуслуги 2024_КД_ДТСЗН"
9,назначение платежа: (512.0703.0130161310.625.241 л/с 02133D06370) л/с 03512140611 Субсидии на иные цели(ПФДО) согласно соглашения №8-10-2023 от 30.10.2023 образ.услуги за 07.24 сторон.орг-ии\nназвание статьи: Субсидии\nназвание проекта: Прочие МО\nкатегория донора: Муниципалитеты


In [4]:
y_pred_ygpt_ft = pd.DataFrame(columns=["universal_category"])

sdk = YCloudML(
        folder_id="YOUR_FOLDER_ID",
        auth="YOUR_YANDEX_CLOUD_TOKEN",
    )

model = sdk.models.text_classifiers(
        "cls://YOUR_FOLDER_ID/yandexgpt-lite/latest@tamros8p69qq7ribmm06k"
    )

for index__, text in tqdm(X_test['text'].items(), total=len(X_test)):
    attempts = 0
    max_attempts = 3
    while attempts < max_attempts:
        try:
            result = model.run(str(text))  # приводим текст к строке
            #print(result, "\n")
            best_label = max(result, key=lambda x: x.confidence).label
            y_pred_ygpt_ft.loc[index__, "universal_category"] = best_label
            break
        except Exception as e:
            attempts += 1
            print(f"Ошибка на index {index__}, попытка {attempts}/{max_attempts}: {e}")
            time.sleep(2)
    else:
        # если все попытки неудачные, присваиваем NaN
        y_pred_ygpt_ft.loc[index__, "universal_category"] = np.  


100%|██████████| 900/900 [30:51<00:00,  2.06s/it]


In [5]:
y_pred_ygpt_ft.to_csv("y_pred_ygpt_ft.csv", index=True, header=["universal_category"])